In [2]:
import requests
import pandas as pd

API='https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total'
HEADERS={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0',
    'Accept':'application/json', # 응답을 json 형식으로 주도록
}

In [ ]:
def fetch(start=1,display=100) -> list[dict]:
    res=requests.get(API,params={
        'start':start,
        'display':display
    }, headers=HEADERS,timeout=10)
    res.raise_for_status()
    data=res.json()
    # data 한번 확인해보고 추출해야할 항목을 정한다
    items=data['response']['result']['chart']['items']['tracks']
    rows=[]

    for i,t in enumerate(items,start=start):
        artist=[a.get('artistName','') for a in t.get('artists',[])]
        rows.append({
            '순위':i,
            '곡명':t.get('trackTitle',''),
            '아티스트':artist,
        })
    return rows


In [ ]:
res=requests.get(API,params={
        'start':1,
        'display':100
    }, headers=HEADERS,timeout=10)
res.raise_for_status()
data=res.json()
data['response']['result']['chart']['items']['tracks'] # 하나씩 단계 내려가기

[{'trackId': 86839869,
  'trackTitle': 'LOVE ATTACK',
  'represent': True,
  'discNumber': 1,
  'trackNumber': 2,
  'artistTotalCount': 1,
  'artists': [{'artistId': 8416557,
    'artistName': 'RESCENE(리센느)',
    'isGroup': True,
    'imageUrl': 'https://musicmeta-phinf.pstatic.net/artist/008/416/8416557.jpg?type=r300&v=20260701100846'}],
  'album': {'albumId': 32037446,
   'albumTitle': 'SCENEDROME',
   'artistTotalCount': 1,
   'artists': [{'artistId': 8416557,
     'artistName': 'RESCENE(리센느)',
     'isGroup': True,
     'imageUrl': 'https://musicmeta-phinf.pstatic.net/artist/008/416/8416557.jpg?type=r300&v=20260701100846'}],
   'releaseDate': '2024-08-27',
   'imageUrl': 'https://musicmeta-phinf.pstatic.net/album/032/037/32037446.jpg?type=r480Fll&v=20260814145821',
   'isAdult': False,
   'albumGenres': '댄스,일렉트로니카',
   'productionName': '카카오엔터테인먼트',
   'agencyName': '더뮤즈엔터테인먼트',
   'isDolbyAtmos': False,
   'hasDolbyAtmos': False,
   'shareUrl': 'https://vibe.naver.com/album/320374

In [9]:
rows=fetch() # accept를 안 주면 json형식이 아님
assert len(rows)==100,f'건수: {len(rows)}'
# assert is a Python simple statement used for debugging: 
# it checks an expression and raises AssertionError if the condition is false.
# assert <expression>,<message>

In [11]:
df=pd.DataFrame(rows)
df['아티스트']=df['아티스트'].map(lambda x:','.join(x))
df.to_csv('vibe_top100.csv',index=False,encoding='utf-8-sig') # euc-kr은 \xc9 문자를 인코딩할 수 없음

In [12]:
print(df.head())

   순위           곡명           아티스트
0   1  LOVE ATTACK   RESCENE(리센느)
1   2          갑자기  아이오아이 (I.O.I)
2   3       REDRED  CORTIS (코르티스)
3   4      여름아 부탁해         볼빨간사춘기
4   5      It's Me     아일릿(ILLIT)


In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from datetime import datetime,timedelta # 현재 날짜 파악용
import time # 딜레이 부여용

In [18]:
URL='https://finance.naver.com/item/sise.naver?code=005930'
HEADERS={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',

}
CODE='005930' # 원하는 종목 코드
# TODAY=datetime.now()

In [20]:
res=requests.get(URL,params={'code':CODE,'page':1},headers=HEADERS,timeout=10)
res.raise_for_status()
print(res.text.find('247,500')) # 정적웹 확인

-1


In [3]:
def fetch(page:int) -> str:
    res=requests.get(URL,params={'code':CODE,'page':page},headers=HEADERS,timeout=10)
    res.raise_for_status()
    return res.text

def get_num(t:str) -> int:
    m=re.search(r'\d+',t.replace(',','').strip())
    return int(m.group()) if m else None

def parse(html:str) -> list[dict]:
    soup=BeautifulSoup(html,'html.parser')
    rows=[]
    for tr in soup.select('table.type2 tr'):
        td=tr.select('td')
        if len(td)<7:
            continue

        date=td[0].text.strip()
        direction=td[2].select_one('em.bu_p').text.strip()
        amount=get_num(td[2].text.strip())
        rows.append({
            '날짜':date,
            '종가':get_num(td[1].text),
            '전일비':f'{direction} {amount}',
            '시가':get_num(td[3].text),
            '고가':get_num(td[4].text),
            '저가':get_num(td[5].text),
            '거래량':get_num(td[6].text),
        })
    return rows



In [4]:
result=[]
for page in range(1,40):
    rows=parse(fetch(page))
    if not rows:
        print(f'{page}는 빈 페이지 - 작업 종료')
        break

    # 현재 날짜로부터 1년 전 파악
    latest=rows[0]['날짜']
    if datetime.strptime(latest,'%Y.%m.%d')<datetime.now()-timedelta(days=365):
        print(f'{latest},1년치 시세 확인 완료')
        break
    result.extend(rows)
    print(f'{page}페이지 - 누적 {len(result)}건 - 최종 {rows[-1]['날짜']}')
    time.sleep(0.7)

1는 빈 페이지 - 작업 종료


In [5]:
df=pd.DataFrame(result).drop_duplicates(subset=['날짜'])
df.to_csv('samsung_1y.csv',index=False,encoding='utf-8-sig')
print(f'\n최종 {len(df)}건 - {df['날짜'].min()}~{df['날짜'].max()}')

KeyError: '날짜'

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

API='https://s.search.naver.com/p/newssearch/3/api/tab/more'
HEADERS={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0',
    'Referer': 'https://search.naver.com/'
}

In [ ]:
def parse(html:str) -> list[dict]:
    soup=BeautifulSoup(html,'html.parser')
    rows=[]
    for item in soup.select(''):
        title=item.select_one('')
        press=item.select_one('')
        summ=item.select_one('')
        if title is None:
            continue
        rows.append({
            '제목':title.get('title') or title.text,
            '언론사':press.text if press else '',
            '링크':title.get('',''),
            '요약':summ.text if summ else '',    
        })